MD# ADVANCE NATURAL LANGUAGE PROCESSING (AML23702) - Lab Exercise 4

**Name:** P Prithvi Raj Kandan &nbsp;|&nbsp; **USN:** 1GA23AI033 &nbsp;|&nbsp; Dept. of AI & ML, Global Academy of Technology

**Aim:** Apply Latent Dirichlet Allocation (LDA) to a small collection of documents and display the top five words of each topic.

In [1]:
%pip install -q scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

DATA = Path("../data")
documents = [ln.strip() for ln in (DATA / "documents_alt.txt").read_text(encoding="utf-8").splitlines() if ln.strip()]
print("Loaded", (DATA / "documents_alt.txt").resolve(), "-", len(documents), "documents")

# Convert text into a document-term matrix.
vectorizer = CountVectorizer(stop_words="english")
doc_term_matrix = vectorizer.fit_transform(documents)

# Discover 3 topics.
lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(doc_term_matrix)

words = vectorizer.get_feature_names_out()

for topic_index, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[-5:][::-1]
    top_words = [words[i] for i in top_indices]
    print(f"Topic {topic_index + 1}: {', '.join(top_words)}")


Loaded /workspace/AAT4/data/documents_alt.txt - 6 documents
Topic 1: match, machine, learning, artificial, intelligence
Topic 2: using, patterns, models, data, learn
Topic 3: medicine, prescribed, treated, doctor, hospital


In [3]:
# Show the strongest topic assignment for each document.
topic_scores = lda.transform(doc_term_matrix)

for i, scores in enumerate(topic_scores):
    print(f"Document {i + 1}: Topic {scores.argmax() + 1} | {documents[i]}")

Document 1: Topic 2 | The team won the football match after scoring two goals.
Document 2: Topic 1 | The player trained daily before the tournament and match.
Document 3: Topic 3 | The doctor treated patients and prescribed medicine in the hospital.
Document 4: Topic 2 | Nurses and doctors work in hospitals to improve patient health.
Document 5: Topic 1 | Python is used for machine learning and artificial intelligence.
Document 6: Topic 2 | Machine learning models learn patterns from data using algorithms.


MD### Second collection (used by the web frontend): cricket, cooking and space
On very small corpora LDA depends strongly on the random seed, so we compare two seeds.

In [4]:
docs = [ln.strip() for ln in (DATA / "documents.txt").read_text(encoding="utf-8").splitlines() if ln.strip()]
print("Loaded", (DATA / "documents.txt").resolve())
cv = CountVectorizer(stop_words="english")
dtm = cv.fit_transform(docs)
vocab = cv.get_feature_names_out()
print("Document-term matrix:", dtm.shape)

def run_lda(seed):
    model = LatentDirichletAllocation(n_components=3, random_state=seed).fit(dtm)
    print(f"--- random_state = {seed} ---")
    for t, comp in enumerate(model.components_, start=1):
        print(f"Topic {t}:", ", ".join(vocab[i] for i in comp.argsort()[-5:][::-1]))
    print("Document -> topic:", [int(k) + 1 for k in model.transform(dtm).argmax(axis=1)])
    print()

run_lda(5)
run_lda(42)


Loaded /workspace/AAT4/data/documents.txt
Document-term matrix: (9, 39)
--- random_state = 5 ---
Topic 1: spices, curry, onions, rice, cook
Topic 2: orbit, earth, space, satellite, track
Topic 3: cricket, match, runs, bowler, wickets
Document -> topic: [3, 3, 3, 1, 1, 1, 2, 2, 2]

--- random_state = 42 ---
Topic 1: spices, curry, onions, rice, cook
Topic 2: orbit, earth, wickets, took, bowler
Topic 3: batsman, scored, team, satellite, runs
Document -> topic: [2, 3, 2, 1, 1, 1, 3, 2, 2]



MD With seed 5 the three topics separate cleanly into cooking, space and cricket. With seed 42 the words of the two groups are mixed, which shows how unstable LDA is on a corpus of only nine sentences.